# 02 — Seed and crawl small

**Goal.** Build a small seed list of onion URLs from Ahmia's search index and fetch each page once, saving the raw HTML and a metadata record per URL.

**Scope.** ~100 pages, synchronous, single Tor circuit per fetch, idempotent (re-running skips URLs already on disk). No link-following — the seeds *are* the crawl.

**Outputs.**
- `data/raw_html/<sha1(url)>.html` — one file per fetched page
- `data/raw_html/_index.jsonl` — one metadata record per fetch attempt

## 1. Imports + the helper from notebook 01

In [3]:
import os, json, time, random, hashlib
from pathlib import Path
import requests
from stem import Signal
from stem.control import Controller

TOR_PROXIES = {"http": "socks5h://127.0.0.1:9050", "https": "socks5h://127.0.0.1:9050"}
USER_AGENT = "Mozilla/5.0 (cti_401 research crawler; contact: you@example.com)"

def new_circuit():
    with Controller.from_port(port=9051) as c:
        c.authenticate()
        c.signal(Signal.NEWNYM)

def safe_fetch(url, *, max_retries=2, timeout=60, min_delay=3.0, max_delay=6.0):
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(max_retries):
        try:
            r = requests.get(url, proxies=TOR_PROXIES, headers=headers, timeout=timeout)
            time.sleep(random.uniform(min_delay, max_delay))
            return r.status_code, r.text
        except Exception as e:
            print(f"  retry {attempt+1}: {type(e).__name__}")
            try:
                new_circuit(); time.sleep(11)
            except Exception:
                pass
    return None, None

## 2. Build a seed list from Ahmia

Ahmia indexes onion services and exposes a search interface over **clearnet** (`https://ahmia.fi/search/?q=...`). We query a few CTI-relevant terms and scrape the result links.

**Allowlist.** The terms below are deliberately narrow. Do not add terms that surface CSAM-adjacent or violent content. If a result snippet looks abusive, drop the URL.

In [4]:
import re
from urllib.parse import urlparse, parse_qs, unquote

AHMIA_TERMS = [
    "leaks", "ransomware", "data breach",
    "cve exploit", "hacking forum", "stealer logs",
]

# Known-public, CTI-relevant or sanity-check onions. Review before adding more.
MANUAL_SEEDS = [
    "http://2gzyxa5ihm7nsggfxnu52rck2vv4rvmdlkiu3zzui5du4xyclen53wid.onion/",      # Tor Project
    "http://juhanurmihxlp77nkq76byazcldy2hlmovfu2epvl5ankdibsot4csyd.onion/",      # Ahmia (onion)
    "http://p53lf57qovyuvwsc6xnrppyply3vtqm7l6pcobkmyqsiofyeznfu5uqd.onion/",      # ProPublica
    "http://bbcnewsd73hkzno2ini43t4gblxvycyac5aw4gnv7t2rccijh7745uqd.onion/",      # BBC News
    "http://sdolvtfhatvsysc6l34d65ymdwxcujausv7k5jk4cy5ttzhjoi6fzvyd.onion/",      # SecureDrop directory
]

def ahmia_search(term, max_results=25):
    """Return onion URLs from Ahmia for `term`, or [] if Ahmia is misbehaving.

    Detects the current failure mode (Ahmia 302-redirects /search/?q=... back
    to /), so we don't waste time regexing the homepage.
    """
    url = f"https://ahmia.fi/search/?q={requests.utils.quote(term)}"
    try:
        r = requests.get(
            url,
            headers={"User-Agent": USER_AGENT},
            timeout=30,
            allow_redirects=False,
        )
    except Exception as e:
        print(f"  ahmia error for {term!r}: {type(e).__name__}: {e}")
        return []

    if 300 <= r.status_code < 400:
        return []
    if r.status_code != 200:
        return []

    hrefs = re.findall(r'href="(/search/redirect[^"]+)"', r.text)
    onions = []
    for h in hrefs:
        target = parse_qs(urlparse(h).query).get("redirect_url", [None])[0]
        if target and ".onion" in target:
            onions.append(unquote(target))
        if len(onions) >= max_results:
            break
    return onions

seed_set = set()
for term in AHMIA_TERMS:
    found = ahmia_search(term, max_results=20)
    print(f"{term!r}: {len(found)} hits")
    seed_set.update(found)
    time.sleep(2)

if not seed_set:
    print("\nAhmia returned nothing (likely a server-side change — clearnet "
          "search currently 302s to homepage). Falling back to manual seed list.")
    seed_set.update(MANUAL_SEEDS)

seeds = sorted(seed_set)
print(f"\ntotal unique onion URLs: {len(seeds)}")

'leaks': 0 hits
'ransomware': 0 hits
'data breach': 0 hits
'cve exploit': 0 hits
'hacking forum': 0 hits
'stealer logs': 0 hits

Ahmia returned nothing (likely a server-side change — clearnet search currently 302s to homepage). Falling back to manual seed list.

total unique onion URLs: 5


If Ahmia returns very few results (it does occasionally), drop in a manual seed list of known CTI-relevant onions. Keep the list small and review each entry.

In [5]:
# Cap at 100 to respect the starter scope
MAX_SEEDS = 100
seeds = seeds[:MAX_SEEDS]
print(f"crawling {len(seeds)} URLs")

crawling 5 URLs


## 3. Crawl, save HTML, append metadata

Filename is `sha1(url).html`. The crawler is idempotent — if the file exists we skip the fetch.

In [6]:
RAW_DIR = Path("data/raw_html")
RAW_DIR.mkdir(parents=True, exist_ok=True)
INDEX = RAW_DIR / "_index.jsonl"

def url_id(url: str) -> str:
    return hashlib.sha1(url.encode()).hexdigest()

def already_fetched(url: str) -> bool:
    return (RAW_DIR / f"{url_id(url)}.html").exists()

In [ ]:
from datetime import datetime, timezone

ok, fail, skipped = 0, 0, 0
with INDEX.open("a") as idx:
    for i, url in enumerate(seeds, 1):
        if already_fetched(url):
            skipped += 1
            continue
        print(f"[{i}/{len(seeds)}] {url[:70]}...")
        status, text = safe_fetch(url)
        record = {
            "url": url,
            "url_id": url_id(url),
            "status": status,
            "bytes": len(text) if text else 0,
            "fetched_at": datetime.now(timezone.utc).isoformat(),
        }
        if status == 200 and text:
            (RAW_DIR / f"{url_id(url)}.html").write_text(text, encoding="utf-8")
            ok += 1
        else:
            fail += 1
        idx.write(json.dumps(record) + "\n")
        idx.flush()

print(f"\nok={ok} fail={fail} skipped={skipped}")

[4/5] http://p53lf57qovyuvwsc6xnrppyply3vtqm7l6pcobkmyqsiofyeznfu5uqd.onion/...
  retry 1: ConnectionError


Expect a substantial failure rate (30–60%). Onion services come and go. That is normal at this scale.

## What's next

Notebook 03 reads `data/raw_html/*.html`, extracts clean text with trafilatura, deduplicates near-identical pages with MinHash, and persists everything to `db/cti.duckdb`.